# Comparação de dois corpus e embeddings — piloto anual

As duas versões usam os mesmos IDs: texto com limpeza mínima e texto lematizado, sem stopwords e com a lista procedural. Isto compara várias alterações em conjunto, não isola o efeito de cada uma. Usa-se o corpus filtrado por papel e anos 2015–2017. Intervenções com menos de 10 palavras são removidas antes da lematização; textos lexicais vazios são excluídos de ambas as versões.


In [1]:
from pathlib import Path
import hashlib, json, re
import numpy as np
import pandas as pd
import spacy

paths = [Path('../../data/parlamint_pt_merged.csv'), Path('data/parlamint_pt_merged.csv')]
input_file = next((p for p in paths if p.exists()), None)
if input_file is None:
    raise FileNotFoundError('Não foi encontrado data/parlamint_pt_merged.csv')
metadata = pd.read_csv(input_file, dtype='string')
metadata['ano'] = pd.to_datetime(metadata['Date'], errors='coerce').dt.year
anos = [2015, 2016, 2017]
metadata = metadata.loc[metadata['ano'].isin(anos)].copy()
if metadata['ID'].isna().any() or metadata['ID'].duplicated().any():
    raise ValueError('Os IDs devem estar preenchidos e ser únicos.')
print('Intervenções iniciais:', len(metadata))


Intervenções iniciais: 51871


## Preparar os dois textos

São reutilizadas as regras do notebook de processamento textual.


In [3]:
def preparar_texto_embeddings(texto):
    if pd.isna(texto):
        return ""
    texto = re.sub(r"\[\[.*?\]\]", " ", str(texto), flags=re.DOTALL)
    return re.sub(r"\s+", " ", texto).strip()

metadata["texto_embeddings"] = metadata["texto"].map(preparar_texto_embeddings)
print(f"Textos vazios após limpeza: {metadata['texto_embeddings'].eq('').sum():,}")

MIN_PALAVRAS = 10
metadata["n_palavras"] = metadata["texto_embeddings"].str.count(r"\b\w+(?:[-’']\w+)*\b")

intervencoes_antes = len(metadata)
metadata = metadata.loc[metadata["n_palavras"] >= MIN_PALAVRAS].copy()
intervencoes_removidas = intervencoes_antes - len(metadata)

print(f"Intervenções antes do filtro: {intervencoes_antes:,}")
print(f"Intervenções removidas: {intervencoes_removidas:,}")
if intervencoes_antes:
    print(f"Percentagem removida: {100 * intervencoes_removidas / intervencoes_antes:.1f}%")
print(f"Intervenções restantes: {len(metadata):,}")

Textos vazios após limpeza: 0
Intervenções antes do filtro: 51,871
Intervenções removidas: 15,610
Percentagem removida: 30.1%
Intervenções restantes: 36,261


In [ ]:
SPACY_MODEL = "pt_core_news_sm"
NEGACOES_A_PRESERVAR = {"não", "nem", "nunca", "jamais", "sem"}
TERMOS_PROCEDIMENTAIS = {
    "sr", "sra", "srs", "sr.ª", "ª", "senhor", "senhora", "senhores", "senhoras",
    "deputado", "deputados", "deputada", "deputadas", "presidente", "vice-presidente",
    "secretário", "secretária", "excelência", "excelências", "exmo", "exma",
    "obrigado", "obrigada", "agradeço", "cumprimento", "cumprimentos",
    "aplausos", "protestos", "risos", "interrupção", "interrupções",
}
BATCH_SIZE = 128

try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
except OSError as error:
    raise OSError(
        f"O modelo spaCy {SPACY_MODEL!r} não está instalado no kernel atual. "
        "Instala-o antes de continuar."
    ) from error

from spacy.matcher import PhraseMatcher
matcher_procedimental = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_procedimental.add("PROCEDIMENTAL", [nlp.make_doc(termo) for termo in sorted(TERMOS_PROCEDIMENTAIS)])

print(f"Modelo linguístico: {SPACY_MODEL}")

def lemas_relevantes(doc):
    lemas = []
    indices_procedimentais = {i for _, inicio, fim in matcher_procedimental(doc) for i in range(inicio, fim)}
    for token in doc:
        if token.i in indices_procedimentais:
            continue
        if token.is_space or token.is_punct or not (token.is_alpha or token.like_num):
            continue
        forma = token.text.lower()
        lema = token.lemma_.strip().lower() or forma
        if forma in TERMOS_PROCEDIMENTAIS or lema in TERMOS_PROCEDIMENTAIS:
            continue
        if token.is_stop and forma not in NEGACOES_A_PRESERVAR and lema not in NEGACOES_A_PRESERVAR:
            continue
        lemas.append(lema)
    return lemas

tokens_lexicais = [
    lemas_relevantes(doc)
    for doc in nlp.pipe(metadata["texto_embeddings"].tolist(), batch_size=BATCH_SIZE)
]
metadata["texto_lexical"] = [" ".join(tokens) for tokens in tokens_lexicais]
metadata["n_tokens_lexicais"] = [len(tokens) for tokens in tokens_lexicais]

print(f"Intervenções sem tokens lexicais: {metadata['n_tokens_lexicais'].eq(0).sum():,}")

In [ ]:
n_antes = len(metadata)
metadata = metadata.loc[metadata['texto_lexical'].str.strip().ne('')].copy()
print(f'Excluídas de ambas as versões por texto lexical vazio: {n_antes - len(metadata):,}')
metadata = metadata.sort_values(['ano', 'ID']).reset_index(drop=True)
print(metadata.groupby('ano').size().to_string())
output_dir = input_file.resolve().parent / 'bertilda_annual' / 'comparacao_textos_2015_2017'
output_dir.mkdir(parents=True, exist_ok=True)
corpus_file = output_dir / 'corpus_comparacao.csv'
if corpus_file.exists():
    raise FileExistsError(f'Já existe {corpus_file}; preservado para evitar substituição.')
metadata[['ID', 'ano', 'texto_embeddings', 'texto_lexical']].to_csv(corpus_file, index=False, encoding='utf-8')
print(f'Corpus guardado: {corpus_file}')


## Gerar embeddings por versão

Executa no kernel com MPS disponível. O modelo é BERTimbau base com mean pooling, 512 tokens e normalização. Ficheiros separados por versão e por bloco permitem retomar após interrupção. A assinatura inclui IDs, texto, modelo e parâmetros; se mudar algum deles, escolhe outra pasta. O modelo trunca textos longos a 512 tokens.


In [ ]:
import torch
from sentence_transformers import SentenceTransformer, models

if not torch.backends.mps.is_available():
    raise RuntimeError('MPS não está disponível neste kernel. Executa esta célula no Mac com um kernel PyTorch com suporte MPS.')
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
MAX_SEQ_LENGTH = 512
BATCH_SIZE = 4
CHUNK_SIZE = 512
DEVICE = 'mps'

transformer = models.Transformer(MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH)
pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode_mean_tokens=True,
                         pooling_mode_cls_token=False, pooling_mode_max_tokens=False)
embedding_model = SentenceTransformer(modules=[transformer, pooling], device=DEVICE)
print(f'Modelo: {MODEL_NAME}; dispositivo: {DEVICE}')


In [ ]:
def assinatura_corpus(ids, textos, versao):
    h = hashlib.sha256()
    config = {'model': MODEL_NAME, 'max_seq_length': MAX_SEQ_LENGTH, 'pooling': 'mean',
              'normalize_embeddings': True, 'version': versao, 'chunk_size': CHUNK_SIZE}
    h.update(json.dumps(config, sort_keys=True).encode())
    for identificador, texto in zip(ids, textos):
        for valor in (identificador, texto):
            dados = valor.encode('utf-8')
            h.update(len(dados).to_bytes(8, 'big'))
            h.update(dados)
    return h.hexdigest()

ids = metadata['ID'].astype(str).tolist()
for versao, coluna in [('contextual', 'texto_embeddings'), ('lexical', 'texto_lexical')]:
    textos = metadata[coluna].astype(str).tolist()
    assinatura = assinatura_corpus(ids, textos, versao)
    pasta = output_dir / f'embeddings_{versao}'
    pasta.mkdir(exist_ok=True)
    manifesto = pasta / 'manifesto.json'
    if manifesto.exists():
        anterior = json.loads(manifesto.read_text(encoding='utf-8'))
        if anterior['assinatura'] != assinatura:
            raise ValueError(f'Corpus ou configuração diferentes em {pasta}; usa uma nova pasta.')
    else:
        if any(pasta.iterdir()):
            raise ValueError(f'Pasta não vazia sem manifesto: {pasta}')
        manifesto.write_text(json.dumps({'assinatura': assinatura, 'modelo': MODEL_NAME,
                                        'versao': versao, 'documentos': len(ids),
                                        'chunk_size': CHUNK_SIZE}, ensure_ascii=False, indent=2), encoding='utf-8')
    for inicio in range(0, len(ids), CHUNK_SIZE):
        fim = min(inicio + CHUNK_SIZE, len(ids))
        destino = pasta / f'bloco_{inicio:08d}_{fim:08d}.npz'
        ids_bloco = np.asarray(ids[inicio:fim], dtype=str)
        if destino.exists():
            with np.load(destino, allow_pickle=False) as bloco:
                if not np.array_equal(bloco['document_ids'], ids_bloco) or bloco['embeddings'].shape[0] != len(ids_bloco):
                    raise ValueError(f'Bloco incompatível: {destino}')
            continue
        embeddings = embedding_model.encode(textos[inicio:fim], batch_size=BATCH_SIZE,
                                             show_progress_bar=True, normalize_embeddings=True)
        temporario = destino.with_suffix('.tmp.npz')
        np.savez_compressed(temporario, document_ids=ids_bloco, embeddings=embeddings)
        temporario.replace(destino)
        print(f'{versao}: {fim:,}/{len(ids):,} documentos guardados')
    print(f'Concluído: {pasta}')


Os blocos guardam IDs na mesma ordem do corpus. Estes ficheiros ainda não são usados automaticamente pelo notebook `bertilda_annual.ipynb`; a ligação ao BERTopic será o passo seguinte.
